In [ ]:
from __future__ import annotations

import json
import math
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
import mlflow
import optuna


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
INNER_CV_FOLDS = 5
TIME_LIMIT_SECONDS = 600.0
MLFLOW_EXPERIMENT_NAME = "Small_Data_Optimization_Ablations"
EARLY_STOP_PATIENCE = 50
EARLY_STOP_MIN_DELTA = 1e-4
rng = np.random.default_rng(RANDOM_STATE)
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print("Imports, plotting style, MLflow experiment, and hold-out plus 5-fold CV protocol configured.")

In [ ]:
def load_sonar_dataset(test_size: float = 0.30, random_state: int = RANDOM_STATE):
    bunch = fetch_openml(name="sonar", version=1, as_frame=True)
    X = bunch.data.select_dtypes(include=[np.number]).copy()
    y_raw = bunch.target.astype(str)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    return {
        "X": X.reset_index(drop=True),
        "y": y,
        "X_train": X_train.reset_index(drop=True),
        "X_test": X_test.reset_index(drop=True),
        "y_train": np.asarray(y_train, dtype=int),
        "y_test": np.asarray(y_test, dtype=int),
        "feature_names": X.columns.tolist(),
        "class_names": label_encoder.classes_.tolist(),
        "label_encoder": label_encoder,
    }


def stratified_subset(X: pd.DataFrame, y: np.ndarray, n_samples: int, random_state: int) -> tuple[pd.DataFrame, np.ndarray]:
    n_samples = int(min(n_samples, len(y)))
    if n_samples >= len(y):
        return X.reset_index(drop=True), np.asarray(y, dtype=int)

    splitter = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=random_state)
    subset_idx, _ = next(splitter.split(X, y))
    return X.iloc[subset_idx].reset_index(drop=True), np.asarray(y[subset_idx], dtype=int)


def make_cv(y: np.ndarray, max_splits: int = 5, random_state: int = RANDOM_STATE) -> StratifiedKFold:
    class_counts = np.bincount(np.asarray(y, dtype=int))
    min_class_count = int(class_counts.min())
    n_splits = max(2, min(max_splits, min_class_count))
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)


DATA = load_sonar_dataset()
X_train_full = DATA["X_train"]
y_train_full = DATA["y_train"]
X_test = DATA["X_test"]
y_test = DATA["y_test"]
feature_names = DATA["feature_names"]
n_total_features = len(feature_names)

print(f"Sonar loaded: train={X_train_full.shape}, test={X_test.shape}, classes={DATA['class_names']}")
print(f"Total features: {n_total_features}")

In [ ]:
def build_pipeline(C: float, gamma: float) -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", C=float(C), gamma=float(gamma))),
    ])


def feature_mask_to_array(mask: np.ndarray | None, n_features: int) -> np.ndarray:
    if mask is None:
        return np.ones(n_features, dtype=bool)
    return np.asarray(mask, dtype=bool)


def subset_features(X: pd.DataFrame, mask: np.ndarray | None) -> pd.DataFrame:
    active_mask = feature_mask_to_array(mask, X.shape[1])
    return X.iloc[:, active_mask].copy()


def evaluate_candidate(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    C: float,
    gamma: float,
    mask: np.ndarray | None,
    alpha: float = 2.0,
    penalty_weight: float = 0.0,
) -> dict[str, Any]:
    X_train_sel = subset_features(X_train, mask)
    X_test_sel = subset_features(X_test, mask)
    cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores: list[float] = []

    for train_idx, val_idx in cv_splitter.split(X_train_sel, y_train):
        X_cv_train = X_train_sel.iloc[train_idx].copy()
        X_cv_val = X_train_sel.iloc[val_idx].copy()
        y_cv_train = np.asarray(y_train[train_idx], dtype=int)
        y_cv_val = np.asarray(y_train[val_idx], dtype=int)

        model = build_pipeline(C, gamma)
        model.fit(X_cv_train, y_cv_train)
        cv_scores.append(float(accuracy_score(y_cv_val, model.predict(X_cv_val))))

    mean_acc = float(np.mean(cv_scores))
    std_acc = float(np.std(cv_scores, ddof=0))
    active_count = int(feature_mask_to_array(mask, X_train.shape[1]).sum())
    feature_fraction = float(active_count / X_train.shape[1])
    fitness = float(mean_acc - alpha * std_acc - penalty_weight * feature_fraction)

    final_model = build_pipeline(C, gamma)
    final_model.fit(X_train_sel, y_train)
    train_acc = float(accuracy_score(y_train, final_model.predict(X_train_sel)))
    test_acc = float(accuracy_score(y_test, final_model.predict(X_test_sel)))

    return {
        "cv_scores": np.asarray(cv_scores, dtype=float),
        "mean_acc": mean_acc,
        "std_acc": std_acc,
        "train_acc": train_acc,
        "test_acc": test_acc,
        "fitness": fitness,
        "active_count": active_count,
        "feature_fraction": feature_fraction,
    }


def repair_mask(mask: np.ndarray) -> np.ndarray:
    repaired = mask.astype(bool).copy()
    if not repaired.any():
        repaired[rng.integers(0, len(repaired))] = True
    return repaired


@dataclass
class Individual:
    C: float
    gamma: float
    mask: np.ndarray | None
    fitness: float = -math.inf
    mean_acc: float = 0.0
    std_acc: float = 0.0
    train_acc: float = 0.0
    test_acc: float = 0.0
    active_count: int = 0
    cv_scores: np.ndarray | None = None
    sigma_c: float = 0.35
    sigma_gamma: float = 0.35
    mask_flip_rate: float = 0.08
    parent_fitness: float = -math.inf

    def clone(self) -> "Individual":
        return Individual(
            C=float(self.C),
            gamma=float(self.gamma),
            mask=None if self.mask is None else self.mask.copy(),
            fitness=float(self.fitness),
            mean_acc=float(self.mean_acc),
            std_acc=float(self.std_acc),
            train_acc=float(self.train_acc),
            test_acc=float(self.test_acc),
            active_count=int(self.active_count),
            cv_scores=None if self.cv_scores is None else self.cv_scores.copy(),
            sigma_c=float(self.sigma_c),
            sigma_gamma=float(self.sigma_gamma),
            mask_flip_rate=float(self.mask_flip_rate),
            parent_fitness=float(self.parent_fitness),
        )


def random_individual(n_features: int, mutation_state: dict, use_mask: bool = True) -> Individual:
    mask = (rng.random(n_features) < 0.5) if use_mask else None
    if mask is not None:
        mask = repair_mask(mask)
    return Individual(
        C=float(10 ** rng.uniform(-3, 3)),
        gamma=float(10 ** rng.uniform(-4, 1)),
        mask=mask,
        sigma_c=float(mutation_state["sigma_c"]),
        sigma_gamma=float(mutation_state["sigma_gamma"]),
        mask_flip_rate=float(mutation_state["mask_flip_rate"]),
    )


def log_space_crossover(value_a: float, value_b: float) -> float:
    return float(np.exp((np.log(value_a) + np.log(value_b)) / 2.0))


def crossover(parent_a: Individual, parent_b: Individual, use_mask: bool = True) -> Individual:
    child_mask = None
    if use_mask:
        assert parent_a.mask is not None and parent_b.mask is not None
        child_mask = np.where(rng.random(len(parent_a.mask)) < 0.5, parent_a.mask, parent_b.mask).astype(bool)
        child_mask = repair_mask(child_mask)
    child = Individual(
        C=log_space_crossover(parent_a.C, parent_b.C),
        gamma=log_space_crossover(parent_a.gamma, parent_b.gamma),
        mask=child_mask,
        sigma_c=float((parent_a.sigma_c + parent_b.sigma_c) / 2.0),
        sigma_gamma=float((parent_a.sigma_gamma + parent_b.sigma_gamma) / 2.0),
        mask_flip_rate=float((parent_a.mask_flip_rate + parent_b.mask_flip_rate) / 2.0),
    )
    return child


def mutate(individual: Individual, mutation_state: dict, mutation_round: int = 1, use_mask: bool = True) -> Individual:
    child = individual.clone()
    child.C = float(np.clip(child.C * np.exp(rng.normal(0.0, child.sigma_c)), 1e-3, 1e3))
    child.gamma = float(np.clip(child.gamma * np.exp(rng.normal(0.0, child.sigma_gamma)), 1e-4, 1e1))

    if use_mask and child.mask is not None:
        flip_mask = rng.random(len(child.mask)) < child.mask_flip_rate
        child.mask = np.logical_xor(child.mask, flip_mask)
        child.mask = repair_mask(child.mask)

    if mutation_round > 0:
        jitter = 1.0 + rng.normal(0.0, 0.02)
        child.sigma_c = float(np.clip(mutation_state["sigma_c"] * jitter, 1e-4, 2.0))
        child.sigma_gamma = float(np.clip(mutation_state["sigma_gamma"] * jitter, 1e-4, 2.0))
        child.mask_flip_rate = float(np.clip(mutation_state["mask_flip_rate"] * jitter, 1e-3, 0.5))

    return child


def initialize_population(pop_size: int, n_features: int, use_mask: bool, mutation_state: dict) -> list[Individual]:
    return [random_individual(n_features, mutation_state, use_mask=use_mask) for _ in range(pop_size)]


def update_mutation_state(mutation_state: dict, success_rate: float) -> dict:
    increase_factor = 1.2
    decrease_factor = 0.85
    factor = increase_factor if success_rate > 0.5 else decrease_factor
    return {
        "mask_flip_rate": float(np.clip(mutation_state["mask_flip_rate"] * factor, 1e-3, 0.5)),
        "sigma_gamma": float(np.clip(mutation_state["sigma_gamma"] * factor, 1e-4, 2.0)),
        "sigma_c": float(np.clip(mutation_state["sigma_c"] * factor, 1e-4, 2.0)),
    }


def tournament_select(population: list[Individual], tournament_size: int = 3) -> Individual:
    contenders = rng.choice(population, size=min(tournament_size, len(population)), replace=False)
    return max(contenders, key=lambda ind: ind.fitness)

print("Core evaluation and EA helpers are ready.")

In [ ]:
def run_ea_search(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    *,
    time_limit: float = TIME_LIMIT_SECONDS,
    pop_size: int = 12,
    elite_size: int = 2,
    use_mask: bool = True,
    alpha: float = 2.0,
    penalty_weight: float = 0.0,
    mutation_sigma: float = 0.25,
    mask_flip_rate: float = 0.08,
    early_stop_patience: int = EARLY_STOP_PATIENCE,
    early_stop_min_delta: float = EARLY_STOP_MIN_DELTA,
    random_state: int = RANDOM_STATE,
) -> dict[str, Any]:

    global rng
    rng = np.random.default_rng(random_state)
    
    mutation_state = {
        "sigma_c": 0.35,
        "sigma_gamma": 0.35,
        "mask_flip_rate": 0.08,
    }
    population = initialize_population(pop_size, X_train.shape[1], use_mask, mutation_state)
    history: list[dict[str, Any]] = []
    start_time = time.time()
    generation = 0
    best_seen_fitness = -math.inf
    generations_since_improvement = 0
    stop_reason = "time_limit"

    while True:
        if time.time() - start_time > time_limit:
            break

        for individual in population:
            if not np.isfinite(individual.fitness):
                metrics = evaluate_candidate(
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    individual.C,
                    individual.gamma,
                    individual.mask,
                    alpha=alpha,
                    penalty_weight=penalty_weight,
                )
                individual.fitness = metrics["fitness"]
                individual.mean_acc = metrics["mean_acc"]
                individual.std_acc = metrics["std_acc"]
                individual.train_acc = metrics["train_acc"]
                individual.test_acc = metrics["test_acc"]
                individual.active_count = metrics["active_count"]
                individual.cv_scores = metrics["cv_scores"]

        population = sorted(population, key=lambda ind: ind.fitness, reverse=True)
        best = population[0]
        current_best_fitness = float(best.fitness)

        if current_best_fitness > best_seen_fitness + early_stop_min_delta:
            best_seen_fitness = current_best_fitness
            generations_since_improvement = 0
        else:
            generations_since_improvement += 1

        history.append({
            "generation": generation + 1,
            "best_fitness": current_best_fitness,
            "best_mean_acc": float(best.mean_acc),
            "best_std_acc": float(best.std_acc),
            "best_train_acc": float(best.train_acc),
            "best_test_acc": float(best.test_acc),
            "best_active_count": int(best.active_count),
            "best_active_pct": float(best.active_count / X_train.shape[1]),
            "sigma_c": float(mutation_state["sigma_c"]),
            "sigma_gamma": float(mutation_state["sigma_gamma"]),
            "mask_flip_rate": float(mutation_state["mask_flip_rate"]),
            "elapsed_sec": float(time.time() - start_time),
            "generations_since_improvement": int(generations_since_improvement),
        })

        if generations_since_improvement >= early_stop_patience:
            stop_reason = "early_stop_plateau"
            break

        if time.time() - start_time > time_limit:
            break

        elites = [ind.clone() for ind in population[:elite_size]]
        next_population = [ind.clone() for ind in elites]
        offspring: list[Individual] = []
        while len(next_population) + len(offspring) < pop_size:
            parent_a = tournament_select(population)
            parent_b = tournament_select(population)
            child = crossover(parent_a, parent_b, use_mask=use_mask)
            child.parent_fitness = float(max(parent_a.fitness, parent_b.fitness))
            child = mutate(child, mutation_state=mutation_state, mutation_round=1, use_mask=use_mask)
            offspring.append(child)

        for child in offspring:
            metrics = evaluate_candidate(
                X_train,
                y_train,
                X_test,
                y_test,
                child.C,
                child.gamma,
                child.mask,
                alpha=alpha,
                penalty_weight=penalty_weight,
            )
            child.fitness = metrics["fitness"]
            child.mean_acc = metrics["mean_acc"]
            child.std_acc = metrics["std_acc"]
            child.train_acc = metrics["train_acc"]
            child.test_acc = metrics["test_acc"]
            child.active_count = metrics["active_count"]
            child.cv_scores = metrics["cv_scores"]

        next_population.extend(offspring)
        success_rate = float(np.mean([child.fitness > child.parent_fitness for child in offspring])) if offspring else 0.0
        mutation_state = update_mutation_state(mutation_state, success_rate)
        population = next_population[:pop_size]
        generation += 1

    best_final = max(population, key=lambda ind: ind.fitness)
    history_df = pd.DataFrame(history)
    return {
        "best_individual": best_final,
        "history": history_df,
        "best_cv_scores": None if best_final.cv_scores is None else best_final.cv_scores.copy(),
        "best_train_acc": float(best_final.train_acc),
        "best_test_acc": float(best_final.test_acc),
        "best_active_count": int(best_final.active_count),
        "best_active_pct": float(best_final.active_count / X_train.shape[1]),
        "generations_completed": int(len(history_df)),
        "elapsed_sec": float(history_df["elapsed_sec"].iloc[-1]) if not history_df.empty else 0.0,
        "stop_reason": stop_reason,
        "best_seen_fitness": float(best_seen_fitness) if np.isfinite(best_seen_fitness) else np.nan,
    }


def run_optuna_search(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    *,
    time_limit: float = TIME_LIMIT_SECONDS,
    early_stop_patience: int = EARLY_STOP_PATIENCE,
    early_stop_min_delta: float = EARLY_STOP_MIN_DELTA,
    random_state: int = RANDOM_STATE,
) -> dict[str, Any]:
    best_seen_value = -math.inf
    trials_since_improvement = 0
    stop_reason = "time_limit"

    def objective(trial: optuna.trial.Trial) -> float:
        C = trial.suggest_float("C", 1e-3, 1e3, log=True)
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)
        inner_cv = StratifiedKFold(n_splits=INNER_CV_FOLDS, shuffle=True, random_state=random_state)
        fold_scores: list[float] = []
        for inner_train_idx, inner_val_idx in inner_cv.split(X_train, y_train):
            X_inner_train = X_train.iloc[inner_train_idx].copy()
            X_inner_val = X_train.iloc[inner_val_idx].copy()
            y_inner_train = np.asarray(y_train[inner_train_idx], dtype=int)
            y_inner_val = np.asarray(y_train[inner_val_idx], dtype=int)
            model = build_pipeline(C, gamma)
            model.fit(X_inner_train, y_inner_train)
            fold_scores.append(float(accuracy_score(y_inner_val, model.predict(X_inner_val))))
        return float(np.mean(fold_scores))

    def plateau_callback(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
        nonlocal best_seen_value, trials_since_improvement, stop_reason

        if trial.value is None:
            return

        if trial.value > best_seen_value + early_stop_min_delta:
            best_seen_value = float(trial.value)
            trials_since_improvement = 0
        else:
            trials_since_improvement += 1

        if trials_since_improvement >= early_stop_patience:
            stop_reason = "early_stop_plateau"
            study.stop()

    sampler = optuna.samplers.TPESampler(seed=random_state)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    start_time = time.time()
    status = "completed"

    try:
        study.optimize(objective, timeout=time_limit, callbacks=[plateau_callback], show_progress_bar=False)
    except Exception as exc:
        status = f"failed: {type(exc).__name__}"

    if len(study.trials) == 0 or study.best_trial is None:
        return {
            "status": status,
            "stop_reason": stop_reason,
            "best_params": None,
            "best_value": np.nan,
            "train_acc": np.nan,
            "test_acc": np.nan,
            "elapsed_sec": float(time.time() - start_time),
            "cv_scores": np.array([]),
            "trials_completed": 0,
        }

    best_params = {
        "C": float(study.best_trial.params["C"]),
        "gamma": float(study.best_trial.params["gamma"]),
    }

    final_model = build_pipeline(best_params["C"], best_params["gamma"])
    final_model.fit(X_train, y_train)
    train_acc = float(accuracy_score(y_train, final_model.predict(X_train)))
    test_acc = float(accuracy_score(y_test, final_model.predict(X_test)))

    if stop_reason == "time_limit" and trials_since_improvement >= early_stop_patience:
        stop_reason = "early_stop_plateau"

    return {
        "status": status,
        "stop_reason": stop_reason,
        "best_params": best_params,
        "best_value": float(study.best_value),
        "train_acc": train_acc,
        "test_acc": test_acc,
        "elapsed_sec": float(time.time() - start_time),
        "cv_scores": np.array([]),
        "trials_completed": int(len(study.trials)),
    }

print("EA and Optuna runners ready.")

In [ ]:
def study_feature_penalty_multi_split(
    penalty_weights: list[float] = [-0.2, -0.05, 0.0, 0.05, 0.2],
    n_splits: int = 3,
    time_limit: float = TIME_LIMIT_SECONDS,
) -> pd.DataFrame:
    X_all = DATA["X"]
    y_all = DATA["y"]
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    detailed_rows = []
    
    with mlflow.start_run(run_name="study_1_feature_penalty_3splits_sonar", nested=False):
        mlflow.log_params({
            "study": "feature_complexity_penalty_multi_split",
            "dataset": "sonar",
            "time_limit_seconds": time_limit,
            "penalty_weights": json.dumps(penalty_weights),
            "n_splits": n_splits,
            "alpha": 2.0,
        })

        for split_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
            X_tr = X_all.iloc[train_idx].reset_index(drop=True)
            y_tr = y_all[train_idx]
            X_te = X_all.iloc[test_idx].reset_index(drop=True)
            y_te = y_all[test_idx]
            
            print(f"\n=== Розбиття {split_idx + 1}/{n_splits} (Train: {len(y_tr)}, Test: {len(y_te)}) ===")

            for penalty in penalty_weights:
                run_name = f"split_{split_idx + 1}_penalty_{penalty:.2f}"
                with mlflow.start_run(run_name=run_name, nested=True):
                    result = run_ea_search(
                        X_tr,
                        y_tr,
                        X_te,
                        y_te,
                        time_limit=time_limit,
                        use_mask=True,
                        alpha=2.0,
                        penalty_weight=penalty,
                        random_state=RANDOM_STATE + split_idx,
                    )
                    
                    row = {
                        "split": split_idx + 1,
                        "penalty_weight": penalty,
                        "test_accuracy": result["best_test_acc"],
                        "active_feature_pct": result["best_active_pct"] * 100.0,
                        "best_fitness": result["history"]["best_fitness"].max() if not result["history"].empty else np.nan,
                        "generations_completed": result["generations_completed"],
                    }
                    detailed_rows.append(row)
                    
                    mlflow.log_metrics({
                        "test_accuracy": float(row["test_accuracy"]),
                        "active_feature_pct": float(row["active_feature_pct"]),
                        "best_fitness": float(row["best_fitness"]),
                        "generations_completed": float(row["generations_completed"]),
                    })
                    mlflow.log_params({"penalty_weight": float(penalty), "split": split_idx + 1})

        raw_df = pd.DataFrame(detailed_rows)
        
        summary = raw_df.groupby("penalty_weight").agg(
            mean_test_acc=("test_accuracy", "mean"),
            std_test_acc=("test_accuracy", "std"),
            mean_active_pct=("active_feature_pct", "mean"),
            std_active_pct=("active_feature_pct", "std"),
            mean_generations=("generations_completed", "mean"),
            mean_fitness=("best_fitness", "mean")
        ).reset_index()

        mlflow.log_text(summary.to_csv(index=False), artifact_file="study_1_summary_averaged.csv")

    fig, ax1 = plt.subplots(figsize=(11, 6))

    ax1.bar(
        summary["penalty_weight"],
        summary["mean_active_pct"],
        yerr=summary["std_active_pct"],
        capsize=5,
        width=0.03,
        color="#4C72B0",
        alpha=0.85,
        label="Active features (%) ± std"
    )
    ax1.set_xlabel("Penalty weight", fontsize=14, labelpad=10)
    ax1.set_ylabel("Active features (%)", color="#4C72B0", fontsize=14)
    ax1.tick_params(axis="y", labelcolor="#4C72B0")
    ax1.set_xticks(summary["penalty_weight"].tolist())
    ax1.set_ylim(0, 100)

    ax2 = ax1.twinx()
    ax2.errorbar(
        summary["penalty_weight"],
        summary["mean_test_acc"],
        yerr=summary["std_test_acc"],
        marker="o",
        linewidth=2.5,
        capsize=5,
        color="#DD8452",
        label="Mean Test accuracy ± std"
    )
    ax2.set_ylabel("Test accuracy", color="#DD8452", fontsize=14)
    ax2.tick_params(axis="y", labelcolor="#DD8452")

    min_val = (summary["mean_test_acc"] - summary["std_test_acc"].fillna(0)).min()
    ax2.set_ylim(max(0.4, min_val - 0.08), 1.02)

    plt.title("Study 1: Feature Complexity Penalty (Averaged across 3 Splits)", pad=15)
    
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc="lower left", frameon=True)
    
    plt.tight_layout()
    plt.show()

    return summary


study1_results = study_feature_penalty_multi_split(time_limit=TIME_LIMIT_SECONDS, n_splits=3)
display(study1_results)

In [ ]:

def study_efficiency_boundary_multi_split(
    sample_sizes: list[int] = [200, 100, 50, 30],
    n_repeats: int = 3,
    time_limit: float = TIME_LIMIT_SECONDS,
) -> pd.DataFrame:
    detailed_rows = []
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    with mlflow.start_run(run_name="study_2_efficiency_multi_split", nested=False):
        mlflow.log_params({
            "study": "efficiency_boundary_multi_split",
            "dataset": "sonar",
            "time_limit_seconds": time_limit,
            "sample_sizes": json.dumps(sample_sizes),
            "n_repeats": n_repeats,
        })

        for requested_n in sample_sizes:
            print(f"\n========== РОЗМІР ВИБІРКИ: {requested_n} ==========")
            
            for rep in range(n_repeats):
                current_seed = RANDOM_STATE + requested_n + rep
                X_sub, y_sub = stratified_subset(X_train_full, y_train_full, requested_n, random_state=current_seed)
                
                with mlflow.start_run(run_name=f"study2_n_{requested_n}_rep_{rep+1}", nested=True):
                    ea_result = run_ea_search(
                        X_sub, y_sub, X_test, y_test,
                        time_limit=time_limit,
                        use_mask=True,
                        alpha=2.0,
                        penalty_weight=0.0,
                        random_state=current_seed,
                    )

                    try:
                        optuna_result = run_optuna_search(
                            X_sub, y_sub, X_test, y_test,
                            time_limit=time_limit,
                            random_state=current_seed,
                        )
                    except Exception as exc:
                        optuna_result = {
                            "status": f"failed: {type(exc).__name__}",
                            "train_acc": np.nan, "test_acc": np.nan, "best_value": np.nan,
                        }

                    optuna_cv_acc = optuna_result.get("best_value", np.nan)
                    optuna_test_acc = optuna_result.get("test_acc", np.nan)
                    
                    ea_cv_acc = float(ea_result["best_individual"].mean_acc)
                    ea_test_acc = ea_result.get("best_test_acc", np.nan)

                    detailed_rows.extend([
                        {
                            "requested_n": requested_n, "repetition": rep + 1, "method": "EA",
                            "cv_accuracy": ea_cv_acc, "test_accuracy": ea_test_acc,
                            "optimization_bias": ea_cv_acc - ea_test_acc,
                        },
                        {
                            "requested_n": requested_n, "repetition": rep + 1, "method": "Optuna",
                            "cv_accuracy": optuna_cv_acc, "test_accuracy": optuna_test_acc,
                            "optimization_bias": optuna_cv_acc - optuna_test_acc,
                        }
                    ])
                    
                    mlflow.log_params({"n_samples": int(requested_n), "repetition": rep + 1})

        raw_df = pd.DataFrame(detailed_rows)
        summary = raw_df.groupby(["requested_n", "method"]).agg(
            mean_cv=("cv_accuracy", "mean"),
            std_cv=("cv_accuracy", "std"),
            mean_test=("test_accuracy", "mean"),
            std_test=("test_accuracy", "std"),
            mean_bias=("optimization_bias", "mean"),
            std_bias=("optimization_bias", "std")
        ).reset_index()

        mlflow.log_text(summary.to_csv(index=False), artifact_file="study_2_summary_averaged.csv")

    plt.figure(figsize=(10, 6))
    
    sns.barplot(
        data=raw_df, 
        x="requested_n", 
        y="optimization_bias", 
        hue="method", 
        palette=["#4C72B0", "#DD8452"],
        errorbar="sd", 
        capsize=0.1,
        edgecolor=".2"
    )
    
    plt.axhline(0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    plt.title("Study 2: Optimization Bias vs Sample Size (Averaged over 3 splits)", pad=15)
    plt.xlabel("Кількість екземплярів у навчальній вибірці (N)")
    plt.ylabel("Generalization Gap (CV Mean - Test Acc)")
    plt.legend(title="Метод")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    sorted_summary = summary.sort_values(["requested_n", "method"], ascending=[False, True]).reset_index(drop=True)
    return sorted_summary, raw_df

study2_results, study2_raw_data = study_efficiency_boundary_multi_split(time_limit=TIME_LIMIT_SECONDS, n_repeats=3)

display(study2_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_learning_curves(summary_df: pd.DataFrame):
    melted = summary_df.melt(
        id_vars=["requested_n", "method"], 
        value_vars=["mean_cv", "mean_test"],
        var_name="split", 
        value_name="accuracy"
    )
    
    melted["split"] = melted["split"].replace({
        "mean_cv": "Оцінка CV (Train)", 
        "mean_test": "Оцінка Test (Hold-out)"
    })
    
    plt.figure(figsize=(11, 7))
    
    sns.lineplot(
        data=melted, 
        x="requested_n", 
        y="accuracy", 
        hue="method", 
        style="split",
        markers=True,
        dashes={"Оцінка CV (Train)": (3, 3), "Оцінка Test (Hold-out)": ""},
        linewidth=2.5,
        palette={"EA": "#4C72B0", "Optuna": "#DD8452"}
    )
    
    plt.title("Динаміка середньої крос-валідаційної та тестової точності на мікровибірках", pad=15)
    plt.xlabel("Розмір тренувальної вибірки (N)", labelpad=10)
    plt.ylabel("Середня точність (Accuracy)", labelpad=10)
    
    plt.legend(title="Метод та Вибірка", loc='lower right')
    
    plt.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

plot_learning_curves(study2_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_learning_curves_with_variance(raw_df: pd.DataFrame):
    melted = raw_df.melt(
        id_vars=["requested_n", "method"], 
        value_vars=["cv_accuracy", "test_accuracy"],
        var_name="split", 
        value_name="accuracy"
    )
    
    melted["split"] = melted["split"].replace({
        "cv_accuracy": "Оцінка CV (Train)", 
        "test_accuracy": "Оцінка Test (Hold-out)"
    })
    
    plt.figure(figsize=(11, 7))
    
    sns.lineplot(
        data=melted, 
        x="requested_n", 
        y="accuracy", 
        hue="method", 
        style="split",
        markers=True,
        dashes={"Оцінка CV (Train)": (3, 3), "Оцінка Test (Hold-out)": ""},
        linewidth=2.5,
        palette={"EA": "#4C72B0", "Optuna": "#DD8452"},
        errorbar="sd",      
        err_style="bars",    
        err_kws={'capsize': 5, 'elinewidth': 1.5}
    )
    
    plt.title(
        "Залежність узагальнюючої здатності алгоритмів оптимізації від обсягу вибірки\n"
        "(Порівняння CV та незалежного тесту, усереднено по 3 розбиттях)", 
        pad=15, 
        fontsize=14
    )
    plt.xlabel("Розмір тренувальної вибірки (N)", labelpad=10)
    plt.ylabel("Точність (Accuracy)", labelpad=10)
    
    plt.legend(title="Метод та Вибірка", loc='lower right')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

plot_learning_curves_with_variance(study2_raw_data)

In [ ]:
def study_variance_impact_multi_split(
    alphas: list[float] = [0, 1, 2, 3],
    n_splits: int = 3,
    time_limit: float = TIME_LIMIT_SECONDS,
) -> pd.DataFrame:
    X_all = DATA["X"]
    y_all = DATA["y"]
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    detailed_rows = []
    box_rows = []
    
    with mlflow.start_run(run_name="study_3_variance_impact_multi_split", nested=False):
        mlflow.log_params({
            "study": "variance_impact_multi_split",
            "dataset": "sonar",
            "time_limit_seconds": time_limit,
            "alphas": json.dumps(alphas),
            "n_splits": n_splits,
        })

        for split_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
            X_tr = X_all.iloc[train_idx].reset_index(drop=True)
            y_tr = y_all[train_idx]
            X_te = X_all.iloc[test_idx].reset_index(drop=True)
            y_te = y_all[test_idx]
            
            print(f"\n=== Розбиття {split_idx + 1}/{n_splits} ===")

            for alpha in alphas:
                with mlflow.start_run(run_name=f"split_{split_idx+1}_alpha_{alpha}", nested=True):
                    print(f"Запуск EA для alpha={alpha}...")
                    result = run_ea_search(
                        X_tr,
                        y_tr,
                        X_te,
                        y_te,
                        time_limit=time_limit,
                        use_mask=True,
                        alpha=float(alpha),
                        penalty_weight=0.0,
                        random_state=RANDOM_STATE + split_idx,
                    )

                    cv_scores = result["best_cv_scores"]
                    mean_cv = float(result["best_individual"].mean_acc)
                    std_cv = float(result["best_individual"].std_acc)
                    test_acc = float(result["best_test_acc"])
                    gens = result["generations_completed"]

                    detailed_rows.append({
                        "split": split_idx + 1,
                        "alpha": alpha,
                        "mean_cv_acc": mean_cv,
                        "std_cv_acc": std_cv,
                        "test_accuracy": test_acc,
                        "generations_completed": gens,
                    })

                    for score in np.asarray(cv_scores, dtype=float):
                        box_rows.append({
                            "split": split_idx + 1,
                            "alpha": alpha,
                            "cv_accuracy": score,
                        })

                    mlflow.log_metrics({
                        "mean_cv_acc": mean_cv,
                        "std_cv_acc": std_cv,
                        "test_accuracy": test_acc,
                        "generations_completed": float(gens),
                    })
                    mlflow.log_params({"alpha": float(alpha), "split": split_idx + 1})

        raw_df = pd.DataFrame(detailed_rows)
        summary = raw_df.groupby("alpha").agg(
            mean_cv=("mean_cv_acc", "mean"),
            std_cv_between_splits=("mean_cv_acc", "std"), 
            mean_internal_std=("std_cv_acc", "mean"),
            mean_test=("test_accuracy", "mean"),
            std_test=("test_accuracy", "std"),
            mean_generations=("generations_completed", "mean")
        ).reset_index()

        mlflow.log_text(summary.to_csv(index=False), artifact_file="study_3_summary_averaged.csv")

    box_df = pd.DataFrame(box_rows)
    
    plt.figure(figsize=(12, 7))
    
    sns.boxplot(data=box_df, x="alpha", y="cv_accuracy", palette="Set2", showfliers=False)
    
    sns.stripplot(data=box_df, x="alpha", y="cv_accuracy", color=".2", size=6, alpha=0.6, jitter=True)
    
    plt.title("Study 3: Variance Penalty and CV Dispersion (Aggregated across 3 Splits)", fontsize=15, pad=15)
    plt.xlabel("Штраф за дисперсію (Alpha)", fontsize=13)
    plt.ylabel("Точність на окремих фолдах (CV accuracy)", fontsize=13)
    
    min_acc = box_df["cv_accuracy"].min()
    plt.ylim(max(0.4, min_acc - 0.05), 1.02)
    
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    return summary.round(4)


study3_results = study_variance_impact_multi_split(time_limit=TIME_LIMIT_SECONDS, n_splits=3)
display(study3_results)

In [ ]:
import matplotlib.pyplot as plt

def plot_cv_error_bars(study3_summary: pd.DataFrame):
    plt.figure(figsize=(10, 6))
    
    plt.errorbar(
        x=study3_summary["alpha"], 
        y=study3_summary["mean_cv"], 
        yerr=study3_summary["mean_internal_std"], 
        fmt='-o',        
        capsize=6,        
        linewidth=2.5, 
        markersize=8,
        color="#4C72B0",
        label="Середня точність CV ± Внутрішня дисперсія фолдів"
    )
    
    plt.errorbar(
        x=study3_summary["alpha"], 
        y=study3_summary["mean_test"], 
        yerr=study3_summary["std_test"], 
        fmt='--s',        
        capsize=6,
        linewidth=2.5, 
        markersize=8,
        color="#DD8452",
        label="Тестова точність ± Розкид між розбиттями"
    )
    
    plt.title("Вплив штрафу за дисперсію (α) на точність та стабільність\n(Усереднено по 3 розбиттях)", pad=15, fontsize=14)
    plt.xlabel("Коефіцієнт штрафу за дисперсію (alpha)", labelpad=10, fontsize=12)
    plt.ylabel("Точність (Accuracy)", labelpad=10, fontsize=12)
    
    plt.xticks(study3_summary["alpha"])
    
    plt.legend(loc='lower left', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

plot_cv_error_bars(study3_results)

In [ ]:
exact_metrics = study3_results[["alpha", "mean_cv", "mean_internal_std", "mean_test", "std_test"]].copy()
exact_metrics = exact_metrics.round(3)
print(exact_metrics.to_markdown(index=False))

In [ ]:
def study_chromosome_vs_speed_multi_split(
    n_splits: int = 3,
    time_limit: float = TIME_LIMIT_SECONDS,
):
    X_all = DATA["X"]
    y_all = DATA["y"]
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    histories = []
    final_metrics = []
    
    with mlflow.start_run(run_name="study_4_chromosome_speed_multi_split", nested=False):
        for split_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
            X_tr = X_all.iloc[train_idx].reset_index(drop=True)
            y_tr = y_all[train_idx]
            X_te = X_all.iloc[test_idx].reset_index(drop=True)
            y_te = y_all[test_idx]
            
            print(f"\n=== Розбиття {split_idx + 1}/{n_splits} ===")

            print("Запуск HPO only (3 genes)...")
            short_space = run_ea_search(
                X_tr, y_tr, X_te, y_te, time_limit=time_limit,
                use_mask=False, alpha=2.0, penalty_weight=0.0,
                random_state=RANDOM_STATE + split_idx,
            )
            s_hist = short_space["history"].copy()
            s_hist["space"] = "HPO only (3 genes)"
            s_hist["split"] = split_idx + 1
            histories.append(s_hist)
            
            final_metrics.append({
                "split": split_idx + 1,
                "space": "HPO only (3 genes)",
                "cv_accuracy": float(short_space["best_individual"].mean_acc),
                "test_accuracy": float(short_space["best_test_acc"]),
                "generations": short_space["generations_completed"]
            })

            print("Запуск HPO + mask (63 genes)...")
            long_space = run_ea_search(
                X_tr, y_tr, X_te, y_te, time_limit=time_limit,
                use_mask=True, alpha=2.0, penalty_weight=0.0,
                random_state=RANDOM_STATE + split_idx,
            )
            l_hist = long_space["history"].copy()
            l_hist["space"] = "HPO + mask (63 genes)"
            l_hist["split"] = split_idx + 1
            histories.append(l_hist)
            
            final_metrics.append({
                "split": split_idx + 1,
                "space": "HPO + mask (63 genes)",
                "cv_accuracy": float(long_space["best_individual"].mean_acc),
                "test_accuracy": float(long_space["best_test_acc"]),
                "generations": long_space["generations_completed"]
            })

        history_df = pd.concat(histories, ignore_index=True)
        metrics_df = pd.DataFrame(final_metrics)
        
        accuracy_summary = metrics_df.groupby("space").agg(
            mean_cv=("cv_accuracy", "mean"),
            std_cv=("cv_accuracy", "std"),
            mean_test=("test_accuracy", "mean"),
            std_test=("test_accuracy", "std"),
            mean_gens=("generations", "mean")
        ).reset_index().round(4)
        
    return history_df, accuracy_summary

history_results, accuracy_results = study_chromosome_vs_speed_multi_split(n_splits=3, time_limit=TIME_LIMIT_SECONDS)

print("\n=== Фінальні показники точності ===")
display(accuracy_results)

In [ ]:
def plot_accuracy_comparison(accuracy_df: pd.DataFrame):
    plt.figure(figsize=(9, 6))
    
    sns.barplot(
        data=accuracy_df, 
        x="space", 
        y="mean_test",
        palette="viridis",
        edgecolor=".2"
    )
    
    plt.errorbar(
        x=accuracy_df["space"], 
        y=accuracy_df["mean_test"], 
        yerr=accuracy_df["std_test"], 
        fmt='none', c='black', capsize=10, linewidth=1.5
    )
    
    plt.title("Порівняння фінальної тестової точності\n(Після вичерпання ліміту часу)", pad=15)
    plt.xlabel("Простір пошуку", labelpad=10)
    plt.ylabel("Середня тестова точність (Test Accuracy)", labelpad=10)
    
    min_acc = accuracy_df["mean_test"].min()
    plt.ylim(max(0.4, min_acc - 0.15), 1.0)
    
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

plot_accuracy_comparison(accuracy_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(11, 6))

sns.lineplot(
    data=study4_results,
    x="generation",
    y="best_test_acc",
    hue="space",
    errorbar="sd", 
    marker="o",
    linewidth=2.5,
    palette="viridis"
)

plt.title("Тестова точність по поколінях", pad=15)
plt.xlabel("Покоління (Generation)", labelpad=10)
plt.ylabel("Тестова точність (Best Test Accuracy)", labelpad=10)

min_acc = study4_results["best_test_acc"].min()
plt.ylim(max(0.4, min_acc - 0.05), 1.02)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(title="Простір пошуку", loc="lower right")
plt.tight_layout()

plt.savefig(ARTIFACT_DIR / "study4_test_accuracy_generations.png", dpi=300)

plt.show()

In [ ]:

exact_study4_metrics = accuracy_results[["space", "mean_test", "std_test", "mean_gens"]].copy()
exact_study4_metrics = exact_study4_metrics.round(3)

print("Точні числові показники для Дослідження 4:")
print(exact_study4_metrics.to_markdown(index=False))